In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# Define path to one EEG file
sample_eeg_file = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered/subject010_Morning.edf"

# Load the EEG file
raw = mne.io.read_raw_edf(sample_eeg_file, preload=True)

# Print available channels
print("\n📡 Available EEG Channels:")
print(raw.info['ch_names'])

# Selected channel for analysis
selected_channel = "Pz-REF"  # Example: Posterior Hot Zone (Dream Imagery)

# Plot raw EEG data (for the selected channel)
raw.plot(picks=[selected_channel], scalings='auto', duration=10, title=f"EEG Signal - {selected_channel}")

# Plot Power Spectral Density (PSD) for the selected channel
raw.plot_psd(fmin=1, fmax=50, picks=[selected_channel], average=True)

# Extract the data for the selected channel for time-series plot
data, times = raw.get_data(picks=[selected_channel], return_times=True)

# Plot the time-series data for the selected channel
plt.figure(figsize=(10, 4))
plt.plot(times, data.T, label=selected_channel)
plt.xlabel("Time (s)")
plt.ylabel("EEG Signal (uV)")
plt.title(f"Time Series Plot of {selected_channel}")
plt.legend()
plt.show()


In [ ]:
# Display Display metadata metadata
print(raw.info)

# Preprocessing 1 file - test

In [ ]:
import mne

# Load the EEG file
raw = mne.io.read_raw_edf(sample_eeg_file, preload=True)

# Check basic info about the data
print(raw.info)

# Optimal channel selection

In [ ]:
# Print the available channels
print("\n📡 Available EEG Channels:")
print(raw.info['ch_names'])

# Select the relevant channels manually
selected_channels = [
    "Pz-REF", "P3-REF", "P4-REF", "O1-REF", "Oz-REF", "O2-REF", # Posterior "Hot Zone" (Visual Imagery)
    "Po3-REF", "Po4-REF", "Po7-REF", "Po8-REF", # Parieto-Occipital (Dream Recall)
    "Fpz-REF", "Fp1-REF", "Fp2-REF", "Fz-REF", "F3-REF", "F4-REF", # Frontal (Dream Memory & Cognition)
    "Af3-REF", "Af4-REF", # Anterior Frontal (Abstract Thought in Dreams)
    "T7-REF", "T8-REF", "Tp7-REF", "Tp8-REF", # Temporal (Speech, Emotion)
    "C3-REF", "C4-REF", "Cp3-REF", "Cp4-REF" # Central (Lucid Dreaming & Sensory Awareness)
    ]


# Drop the channels that are not in the selected list
raw.pick_channels(selected_channels)

# Check the selected channels
print("\nSelected EEG Channels:")
print(raw.info['ch_names'])


# Re-referencing

In [ ]:
# Apply the Common Average Reference (CAR)
raw.set_eeg_reference(ref_channels='average')

# Check the reference status
print("\n✅ EEG re-referenced using the Common Average Reference (CAR).")
print("Reference channels:", raw.info['bads'])  # Check bad channels (if any)

# Band Pass Filtering

In [ ]:
# Apply a band-pass filter: 1-45 Hz (common range for EEG analysis)
raw.filter(l_freq=1, h_freq=45, fir_design='firwin')

# Check the filter status (optional)
print("\n✅ Band-pass filter applied: 1-45 Hz")

# Save file in .fif

In [ ]:
import os

# Define the folder path
output_folder = '/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_process_TEST/'

# Check if the folder exists, and create it if it doesn't
os.makedirs(output_folder, exist_ok=True)

# Save the raw object to the new directory
raw.save(os.path.join(output_folder, 'subject010_Morning.fif'), overwrite=True)

In [ ]:
print(raw.info['ch_names'])

# Frequency extraction for PSD 

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_welch  # Use psd_array_welch for newer versions of MNE
import pandas as pd

# Set frequency bands of interest
freq_bands = {
    'Delta': (1, 4),     # Delta frequency range (0.5-4 Hz)
    'Theta': (4, 8),     # Theta frequency range (4-8 Hz)
    'Alpha': (8, 13),    # Alpha frequency range (8-13 Hz)
    'Beta': (13, 30),    # Beta frequency range (13-30 Hz)
    'Gamma': (30, 40)    # Gamma frequency range (30-40 Hz)
}

# Calculate the PSD using Welch's method
# Ensure raw_clean is a valid MNE Raw object (this should already be cleaned and preprocessed)
psds, freqs = psd_array_welch(raw.get_data(), sfreq=raw.info['sfreq'], fmin=1, fmax=50, n_fft=2048)

# Z-score normalization: Normalize each channel's PSD values across frequency bins
psds_normalized = (psds - np.mean(psds, axis=1, keepdims=True)) / np.std(psds, axis=1, keepdims=True)

# Initialize a dictionary to store band powers
band_powers = {}

# Calculate the band power for each frequency band
for band_name, band_range in freq_bands.items():
    band_psd = psds_normalized[:, (freqs >= band_range[0]) & (freqs <= band_range[1])]
    band_powers[band_name] = band_psd.mean(axis=1)  # Calculate mean power for each channel

# Output band power for each channel
for ch_idx, ch_name in enumerate(raw.info['ch_names']):
    print(f"Channel: {ch_name}")
    for band_name in freq_bands.keys():
        print(f"  Average {band_name} band power (Z-score normalized): {band_powers[band_name][ch_idx]:.4f}")

    # Optionally: Visualize the normalized PSD for each channel
    plt.figure(figsize=(10, 6))
    plt.plot(freqs, psds_normalized[ch_idx], label='Total PSD (Z-score normalized)')
    for band_name, band_range in freq_bands.items():
        plt.axvline(band_range[0], color='r', linestyle='--', label=f'{band_name} start')
        plt.axvline(band_range[1], color='r', linestyle='--', label=f'{band_name} end')

    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Normalized Power Spectral Density (Z-score)')
    plt.title(f'Normalized PSD for {ch_name}')
    plt.legend()
    plt.show()

# Convert the band power dictionary to a DataFrame for easy analysis or saving
band_power_df = pd.DataFrame(band_powers, index=raw.info['ch_names'])
print(band_power_df)


# Downsampling (if needed - test)

In [ ]:
# Downsample the data to 250 Hz (or any rate you prefer)
raw.resample(250, npad="auto")  # You can specify the target sampling rate here

# Check the new sampling rate
print(f"New sampling rate: {raw.info['sfreq']} Hz")


# full pipeline

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_welch
from pathlib import Path

# Path to the folder with EEG files
input_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered"
output_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered_processed_complete"

# Create output directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define frequency bands of interest
freq_bands = {
    'Delta': (1, 4),     # Delta frequency range (0.5-4 Hz)
    'Theta': (4, 8),     # Theta frequency range (4-8 Hz)
    'Alpha': (8, 13),    # Alpha frequency range (8-13 Hz)
    'Beta': (13, 30),    # Beta frequency range (13-30 Hz)
    'Gamma': (30, 40)    # Gamma frequency range (30-40 Hz)
}

# Function to preprocess each EEG file
def preprocess_eeg(file_path):
    # Load the EEG file
    raw = mne.io.read_raw_edf(file_path, preload=True)

    # Select the relevant channels manually
    selected_channels = [
        "Pz-REF", "P3-REF", "P4-REF", "O1-REF", "Oz-REF", "O2-REF", # Posterior "Hot Zone" (Visual Imagery)
        "Po3-REF", "Po4-REF", "Po7-REF", "Po8-REF", # Parieto-Occipital (Dream Recall)
        "Fpz-REF", "Fp1-REF", "Fp2-REF", "Fz-REF", "F3-REF", "F4-REF", # Frontal (Dream Memory & Cognition)
        "Af3-REF", "Af4-REF", # Anterior Frontal (Abstract Thought in Dreams)
        "T7-REF", "T8-REF", "Tp7-REF", "Tp8-REF", # Temporal (Speech, Emotion)
        "C3-REF", "C4-REF", "Cp3-REF", "Cp4-REF" # Central (Lucid Dreaming & Sensory Awareness)
    ]
    
    # Drop the channels that are not in the selected list
    raw.pick_channels(selected_channels)
    
    # Apply the Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average')

    # Apply a band-pass filter: 1-45 Hz (common range for EEG analysis)
    raw.filter(l_freq=1, h_freq=45, fir_design='firwin')

    # Downsample the data to 250 Hz
    raw.resample(250, npad="auto")
    
    # Return the processed raw data
    return raw

# Function to calculate band power
def calculate_band_power(raw):
    psds, freqs = psd_array_welch(raw.get_data(), sfreq=raw.info['sfreq'], fmin=1, fmax=50, n_fft=2048)
    
    # Z-score normalization
    psds_normalized = (psds - np.mean(psds, axis=1, keepdims=True)) / np.std(psds, axis=1, keepdims=True)
    
    # Initialize a dictionary to store band powers
    band_powers = {}
    
    for band_name, band_range in freq_bands.items():
        band_psd = psds_normalized[:, (freqs >= band_range[0]) & (freqs <= band_range[1])]
        band_powers[band_name] = band_psd.mean(axis=1)
    
    return band_powers

# Function to process and save each EEG file
def process_all_files(input_folder, output_folder):
    # Iterate over all EEG files in the input directory
    for eeg_file in Path(input_folder).glob("*.edf"):  # Adjust the extension if necessary
        print(f"Processing file: {eeg_file}")
        
        # Preprocess the EEG data
        raw = preprocess_eeg(eeg_file)
        
        # Calculate band power
        band_powers = calculate_band_power(raw)
        
        # Save processed file
        output_file = os.path.join(output_folder, eeg_file.stem + "_processed.fif")
        raw.save(output_file, overwrite=True)
        
        # Optionally, print or save the band powers
        band_power_df = pd.DataFrame(band_powers, index=raw.info['ch_names'])
        print(band_power_df)
        
        # Visualize the normalized PSD for the first channel (or all channels if needed)
        plt.figure(figsize=(10, 6))
        plt.plot(freqs, psds_normalized[0], label='Total PSD (Z-score normalized)')
        for band_name, band_range in freq_bands.items():
            plt.axvline(band_range[0], color='r', linestyle='--', label=f'{band_name} start')
            plt.axvline(band_range[1], color='r', linestyle='--', label=f'{band_name} end')
        
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Normalized Power Spectral Density (Z-score)')
        plt.title(f'Normalized PSD for {raw.info["ch_names"][0]}')
        plt.legend()
        plt.show()

# Run the pipeline on all files
process_all_files(input_folder, output_folder)


# full pipeline with csv + flattened vec

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_welch
from pathlib import Path

# Path to the folder with EEG files
input_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered"
output_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered_processed_complete1"

# Create output directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define frequency bands of interest
freq_bands = {
    'Delta': (1, 4),     # Delta frequency range (0.5-4 Hz)
    'Theta': (4, 8),     # Theta frequency range (4-8 Hz)
    'Alpha': (8, 13),    # Alpha frequency range (8-13 Hz)
    'Beta': (13, 30),    # Beta frequency range (13-30 Hz)
    'Gamma': (30, 40)    # Gamma frequency range (30-40 Hz)
}

# Function to preprocess each EEG file
def preprocess_eeg(file_path):
    # Load the EEG file
    raw = mne.io.read_raw_edf(file_path, preload=True)

    # Select the relevant channels manually
    selected_channels = [
        "Pz-REF", "P3-REF", "P4-REF", "O1-REF", "Oz-REF", "O2-REF", # Posterior "Hot Zone" (Visual Imagery)
        "Po3-REF", "Po4-REF", "Po7-REF", "Po8-REF", # Parieto-Occipital (Dream Recall)
        "Fpz-REF", "Fp1-REF", "Fp2-REF", "Fz-REF", "F3-REF", "F4-REF", # Frontal (Dream Memory & Cognition)
        "Af3-REF", "Af4-REF", # Anterior Frontal (Abstract Thought in Dreams)
        "T7-REF", "T8-REF", "Tp7-REF", "Tp8-REF", # Temporal (Speech, Emotion)
        "C3-REF", "C4-REF", "Cp3-REF", "Cp4-REF" # Central (Lucid Dreaming & Sensory Awareness)
    ]
    
    # Drop the channels that are not in the selected list
    raw.pick_channels(selected_channels)
    
    # Apply the Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average')

    # Apply a band-pass filter: 1-45 Hz (common range for EEG analysis)
    raw.filter(l_freq=1, h_freq=45, fir_design='firwin')

    # Downsample the data to 250 Hz
    raw.resample(250, npad="auto")
    
    # Return the processed raw data
    return raw

# Function to calculate band power and flatten it
def calculate_flattened_band_power(raw):
    psds, freqs = psd_array_welch(raw.get_data(), sfreq=raw.info['sfreq'], fmin=1, fmax=50, n_fft=2048)
    
    # Z-score normalization
    psds_normalized = (psds - np.mean(psds, axis=1, keepdims=True)) / np.std(psds, axis=1, keepdims=True)
    
    # Initialize a list to store the flattened band powers for each channel
    flattened_band_powers = []
    
    for band_name, band_range in freq_bands.items():
        band_psd = psds_normalized[:, (freqs >= band_range[0]) & (freqs <= band_range[1])]
        flattened_band_powers.extend(band_psd.mean(axis=1))  # Flatten and add the mean band power for each channel
    
    return np.array(flattened_band_powers)  # Return as a 1D array of 130 features

# Function to process all EEG files and save features
def process_all_files(input_folder, output_folder):
    # Initialize a list to store EEG features and their corresponding file names
    eeg_features = []
    filenames = []

    # Iterate over all EEG files in the input directory
    for eeg_file in Path(input_folder).glob("*.edf"):  # Adjust the extension if necessary
        print(f"Processing file: {eeg_file}")
        
        # Preprocess the EEG data
        raw = preprocess_eeg(eeg_file)
        
        # Calculate and flatten the band power
        flattened_band_powers = calculate_flattened_band_power(raw)
        
        # Append the flattened features and filename
        eeg_features.append(flattened_band_powers)
        filenames.append(eeg_file.stem)

        # Save the processed EEG data to output folder
        output_file = os.path.join(output_folder, eeg_file.stem + "_processed.fif")
        raw.save(output_file, overwrite=True)
    
    # Convert the EEG features to a DataFrame for easy handling
    eeg_features_df = pd.DataFrame(eeg_features)
    eeg_features_df['filename'] = filenames
    
    # Optionally save the DataFrame for further processing
    eeg_features_df.to_csv(os.path.join(output_folder, 'eeg_features.csv'), index=False)
    print(f"EEG features saved to {output_folder}/eeg_features.csv")

# Run the pipeline on all files
process_all_files(input_folder, output_folder)


# Merging EEG features + report embeddings

In [ ]:
import pandas as pd
import torch

# Load EEG features (already reordered)
eeg_features = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/eeg_features_reordered.csv")  # ← your actual path

# Load embeddings
embeddings = torch.load("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Embeddings.pt")  # ← your actual path

# Convert embeddings to a pandas DataFrame
embeddings_df = pd.DataFrame(embeddings.numpy())

# Merge EEG features and embeddings side by side
merged_df = pd.concat([eeg_features, embeddings_df], axis=1)

# Save the merged DataFrame
merged_df.to_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/merged_eeg_embeddings.csv", index=False)

print("✅ Successfully merged EEG features and embeddings!")


In [ ]:
print(merged_df.shape)

In [ ]:
import pandas as pd

# Load the CSV file
merged_eeg_embeddings = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/merged_eeg_embeddings.csv")

# Check the shape of the DataFrame
print(merged_eeg_embeddings.shape)